# Raspador da ANVISA (Agência Nacional de Vigilância Sanitária)

Este notebook demonstra o uso do raspador para buscar atos normativos da Agência Nacional de Vigilância Sanitária no portal ANVISALegis.

**Fonte:** [anvisalegis.datalegis.net](https://anvisalegis.datalegis.net/)

**Tipo de dados:** Resoluções da Diretoria Colegiada (RDC), instruções normativas, portarias, extratos e demais atos da ANVISA

**Requisitos:** Este raspador usa Playwright para automação de navegador.

```bash
pip install "raspe[browser] @ git+https://github.com/bdcdo/raspe.git"
python -m playwright install chromium
```

## Importação

In [1]:
import raspe

## Uso Básico

O termo de busca vai no parâmetro `pesquisa`. O nome antigo `termo` ainda funciona, com aviso de descontinuação.

O raspador abre um Chromium sem janela, faz a busca no portal e percorre todas as páginas de resultado, até o limite de 100. Por isso não aceita `paginas`, e cada chamada leva algumas dezenas de segundos.

In [2]:
# Busca simples por um termo
# debug=False: log só no nível INFO, e os HTMLs baixados são apagados ao fim
scraper = raspe.anvisa(debug=False)
dados = scraper.raspar(pesquisa="doença rara")

print(f"Total de atos encontrados: {len(dados)}")
dados.head()

2026-11-15 15:12:52,150 - raspe.ANVISA - INFO - Iniciando raspagem Playwright: {'pesquisa': 'doença rara'}


2026-11-15 15:12:52,644 - raspe.ANVISA - INFO - Browser Playwright iniciado com stealth


2026-11-15 15:12:55,002 - raspe.ANVISA - INFO - Verificando proteção Cloudflare...


2026-11-15 15:12:55,279 - raspe.ANVISA - INFO - Página carregada (sem proteção Cloudflare)


2026-11-15 15:13:14,463 - raspe.ANVISA - INFO - Busca executada


2026-11-15 15:13:18,514 - raspe.ANVISA - INFO - Total de páginas: 1


... (3 outputs left out)

Processando documentos: 100%|██████████| 1/1 [00:00<00:00,  9.92it/s]


2026-11-15 15:13:18,693 - raspe.ANVISA - INFO - Raspagem concluída: 42 registros


Total de atos encontrados: 42


,url,titulo,descricao,situacao,termo_busca
0,https://anvisalegis.datalegis.net/action/Actio...,"Resolução da Diretoria Colegiada - RDC nº 205,...",Estabelece procedimento especial para anuência...,Vigente com Alterações,doença rara
1,https://anvisalegis.datalegis.net/action/Actio...,"Resolução da Diretoria Colegiada - RDC nº 811,...",Dispõe sobre a alteração da Resolução da Diret...,Alterador,doença rara
2,https://anvisalegis.datalegis.net/action/Actio...,Extrato de Compromisso,"Extrato de Termo de Compromisso, celebrado ent...",Vigente,doença rara
3,https://anvisalegis.datalegis.net/action/Actio...,"Resolução da Diretoria Colegiada - RDC nº 16, ...","Altera a Resolução - RDC nº 28, de 4 de abril ...",REVOGADO,doença rara
4,https://anvisalegis.datalegis.net/action/Actio...,VOTO Nº 64/2024/SEI/DIRE2/ANVISA,VOTO Nº 64/2024/SEI/DIRE2/ANVISA - Processo nº...,Vigente,doença rara


## Parâmetros Disponíveis

| Parâmetro | Tipo | Descrição |
|-----------|------|------------|
| `pesquisa` | str | Termo de busca; um termo por chamada |
| `headless` | bool | Do construtor (`raspe.anvisa(headless=False)`); False mostra a janela do navegador |
| `debug` | bool | Do construtor; se True (padrão), registra logs de depuração e mantém os HTMLs baixados |

## Colunas Retornadas

| Coluna | Descrição |
|--------|------------|
| `url` | Link para o texto do ato no ANVISALegis |
| `titulo` | Tipo, número e data do ato |
| `descricao` | Ementa do ato |
| `situacao` | Situação do ato (Vigente, Revogado, Alterador etc.) |
| `termo_busca` | Termo de busca que gerou a linha |

Tipos e exemplos de cada coluna estão em [`docs/colunas.md`](../docs/colunas.md#raspeans-e-raspeanvisa).

## Uso Avançado

In [3]:
# Buscar outro termo e juntar ao resultado anterior: uma chamada por termo
import pandas as pd

dados_orfao = scraper.raspar(pesquisa="medicamento órfão")
dados_multiplos = pd.concat([dados, dados_orfao], ignore_index=True)

# A coluna 'termo_busca' indica qual termo gerou cada resultado
print(dados_multiplos.groupby("termo_busca").size())

# O mesmo ato pode aparecer nas duas buscas
dados_unicos = dados_multiplos.drop_duplicates(subset="url")
print(f"Total de atos únicos: {len(dados_unicos)}")

2026-11-15 15:13:18,703 - raspe.ANVISA - INFO - Iniciando raspagem Playwright: {'pesquisa': 'medicamento órfão'}


2026-11-15 15:13:19,056 - raspe.ANVISA - INFO - Browser Playwright iniciado com stealth


2026-11-15 15:13:21,367 - raspe.ANVISA - INFO - Verificando proteção Cloudflare...


2026-11-15 15:13:21,453 - raspe.ANVISA - INFO - Página carregada (sem proteção Cloudflare)


2026-11-15 15:13:40,350 - raspe.ANVISA - INFO - Busca executada


2026-11-15 15:13:44,408 - raspe.ANVISA - INFO - Total de páginas: 1


2026-11-15 15:13:44,502 - raspe.ANVISA - INFO - Browser encerrado


Processando documentos:   0%|          | 0/1 [00:00<?, ?it/s]

Processando documentos: 100%|██████████| 1/1 [00:00<00:00, 10.84it/s]


2026-11-15 15:13:44,598 - raspe.ANVISA - INFO - Raspagem concluída: 6 registros


termo_busca
doença rara          42
medicamento órfão     6
dtype: int64
Total de atos únicos: 43


In [4]:
# Analisar distribuição por situação do ato
print("Distribuição por situação:")
print(dados["situacao"].value_counts(dropna=False))

Distribuição por situação:
situacao
Vigente                   21
REVOGADO                  12
Vigente com Alterações     6
Alterador                  2
NaN                        1
Name: count, dtype: int64


In [5]:
# Extrair do título o tipo e o ano do ato
# Ex.: "Resolução da Diretoria Colegiada - RDC nº 205, de 28/12/2017" e "VOTO Nº 64/2024/SEI/DIRE2/ANVISA"
dados["tipo_ato"] = dados["titulo"].str.extract(r"^(\w+)", expand=False).str.capitalize()
dados["ano"] = dados["titulo"].str.extract(r"\b((?:19|20)\d{2})\b", expand=False)

print("Distribuição por tipo (primeira palavra do título):")
print(dados["tipo_ato"].value_counts())
print()
print(f"Atos por ano ({dados['ano'].isna().sum()} sem ano no título):")
print(dados["ano"].value_counts().sort_index())

Distribuição por tipo (primeira palavra do título):
tipo_ato
Resolução    19
Voto         16
Instrução     3
Extrato       2
Circuito      1
Portaria      1
Name: count, dtype: int64

Atos por ano (2 sem ano no título):
ano
2003     1
2004     1
2007     1
2008     1
2009     1
2010     1
2013     1
2014     2
2016     1
2017     2
2020     3
2021     1
2022     2
2023     1
2024    16
2025     5
Name: count, dtype: int64


## Exportação dos Dados

In [6]:
# Exportar para Excel
# dados.to_excel("anvisa_resultados.xlsx", index=False)

# Exportar para CSV
# dados.to_csv("anvisa_resultados.csv", index=False)

## Notas

- Este raspador usa Playwright para automação de navegador, então é mais lento que os raspadores HTTP.
- O Chromium precisa estar instalado (`python -m playwright install chromium`).
- Para ver o navegador durante a coleta, use `raspe.anvisa(headless=False)`.